<a href="https://colab.research.google.com/github/Mhmd-Seyed/ecommerce-behavior-analytics/blob/main/notebooks/02_sessions_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

project_files = os.listdir(project_path)

for file in project_files:
    print(file)

In [ ]:
conversion_df.toPandas().to_csv(
    "/content/drive/MyDrive/ecommerce_project/conversion_metrics.csv",
    index=False
)

print("Conversion metrics saved successfully!")

In [ ]:
conversion_rows = [
    ("1_View", int(view_sessions), 100.0, round(view_to_cart_rate, 2)),

    ("2_Cart", int(view_to_cart_sessions),
     round(view_to_cart_rate, 2),
     round(cart_to_purchase_rate, 2)),

    ("3_Purchase", int(full_funnel_purchase),
     round((full_funnel_purchase / view_sessions) * 100, 2),
     100.0)
]

conversion_df = spark.createDataFrame(
    conversion_rows,
    ["stage", "sessions_count", "pct_of_initial_view", "step_conversion_rate"]
)

conversion_df.show()

In [ ]:
import os

output_path = "/content/drive/MyDrive/ecommerce_project/sessions_summary.parquet"

print("Exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("Files:", os.listdir(output_path))

In [ ]:
!cp -r /content/sessions_summary.parquet /content/drive/MyDrive/ecommerce_project/

In [ ]:

# Save session summary as Parquet
sessions_df.write \
    .mode("overwrite") \
    .parquet("/content/sessions_summary.parquet")

print("Parquet file saved successfully!")


In [ ]:

sessions_df.groupBy("session_outcome").count().show()


In [ ]:

# 2. Dönüşüm oranlarını (%) hesapla

view_to_cart_rate = (
    view_to_cart_sessions / view_sessions
) * 100 if view_sessions > 0 else 0

cart_to_purchase_rate = (
    full_funnel_purchase / view_to_cart_sessions
) * 100 if view_to_cart_sessions > 0 else 0

overall_conversion_rate = (
    any_purchase_sessions / total_sessions
) * 100 if total_sessions > 0 else 0

cart_abandonment_rate = 100 - cart_to_purchase_rate

print("=== OTURUM BAZLI DÖNÜŞÜM RAPORU ===")

print(f"Toplam Oturum Sayısı: {total_sessions:,}")
print(f"1. Görüntüleme Oturumları (View): {view_sessions:,}")

print(f"2. Sepete Ekleme (View -> Cart): {view_to_cart_sessions:,} (%{view_to_cart_rate:.2f})")

print(f"3. Satın Alma (Cart -> Purchase): {full_funnel_purchase:,} (%{cart_to_purchase_rate:.2f})")

print(f"Genel Mağaza Dönüşüm Oranı: %{overall_conversion_rate:.2f}")

print(f"Sepet Terk Oranı (Abandonment): %{cart_abandonment_rate:.2f}")


In [ ]:

print("Total sessions:", total_sessions)
print("View sessions:", view_sessions)
print("View → Cart sessions:", view_to_cart_sessions)
print("View → Cart → Purchase sessions:", full_funnel_purchase)
print("Any purchase sessions:", any_purchase_sessions)


In [ ]:

# Hızlı hesaplama için oturum tablosunu belleğe al
sessions_df.cache()

# 1. Aşama sayılarını hesapla
total_sessions = sessions_df.count()

view_sessions = sessions_df.filter(
    F.col("has_view") == 1
).count()

view_to_cart_sessions = sessions_df.filter(
    (F.col("has_view") == 1) &
    (F.col("has_cart") == 1)
).count()

full_funnel_purchase = sessions_df.filter(
    (F.col("has_view") == 1) &
    (F.col("has_cart") == 1) &
    (F.col("has_purchase") == 1)
).count()

any_purchase_sessions = sessions_df.filter(
    F.col("has_purchase") == 1
).count()

print("Session counts calculated successfully!")


In [ ]:
sessions_df.select(
    "user_id",
    "view_count",
    "cart_count",
    "purchase_count",
    "session_duration_sec",
    "has_view",
    "has_cart",
    "has_purchase",
    "session_outcome"
).show(10, truncate=False)

In [ ]:

from pyspark.sql import functions as F

sessions_df = sessions_df \
    .withColumn(
        "session_duration_sec",
        F.unix_timestamp("session_end") -
        F.unix_timestamp("session_start")
    ) \
    .withColumn(
        "has_view",
        F.when(F.col("view_count") > 0, 1).otherwise(0)
    ) \
    .withColumn(
        "has_cart",
        F.when(F.col("cart_count") > 0, 1).otherwise(0)
    ) \
    .withColumn(
        "has_purchase",
        F.when(F.col("purchase_count") > 0, 1).otherwise(0)
    ) \
    .withColumn(
        "session_outcome",
        F.when(F.col("has_purchase") == 1, "purchased")
         .when(F.col("has_cart") == 1, "cart_abandoned")
         .otherwise("view_only")
    )

print("Session features created successfully!")


In [ ]:
sessions_df.show(5, truncate=False)

In [ ]:

from pyspark.sql import functions as F

sessions_df = df.groupBy("user_id", "user_session").agg(
    F.min("event_time").alias("session_start"),
    F.max("event_time").alias("session_end"),
    F.first("event_hour").alias("session_start_hour"),
    F.first("main_category").alias("primary_category"),
    F.sum("is_view").alias("view_count"),
    F.sum("is_cart").alias("cart_count"),
    F.sum("is_purchase").alias("purchase_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.round(F.avg("price"), 2).alias("avg_product_price"),
    F.round(
        F.sum(
            F.when(F.col("event_type") == "purchase", F.col("price"))
            .otherwise(0)
        ), 2
    ).alias("total_spent")
)

print("Session aggregation completed!")


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Student2_Sessions_And_Conversion") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

df = spark.read.parquet(data_path)

print("Data loaded successfully!")
df.printSchema()
df.show(5, truncate=False)

In [ ]:
import os

data_path = os.path.join(project_path, "events_clean.parquet")

print("Data folder exists:", os.path.exists(data_path))
print("Is directory:", os.path.isdir(data_path))

if os.path.isdir(data_path):
    print("Files inside:")
    print(os.listdir(data_path))

In [ ]:
print(os.listdir(project_path))

In [ ]:
import os

project_path = "/content/drive/MyDrive/ecommerce_project"

print("Folder exists:", os.path.exists(project_path))

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

print("Drive mounted:", os.path.exists("/content/drive"))
print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))

if os.path.exists("/content/drive/MyDrive"):
    print("\nMyDrive contents:")
    print(os.listdir("/content/drive/MyDrive"))

In [ ]:
!ls -lh /content/drive/MyDrive/ecommerce_project

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcommerceBehaviorAnalytics") \
    .getOrCreate()

print("Spark Version:", spark.version)

spark.range(5).show()

In [ ]:
import importlib.util

spec = importlib.util.find_spec("pyspark")

if spec is not None:
    print("PySpark is installed")
else:
    print("PySpark is not installed")

In [ ]:
import os

project_path = "/content/drive/MyDrive/ecommerce_project"

print(os.path.exists(project_path))
print(os.listdir(project_path))

In [ ]:
import os

path = '/content/drive/MyDrive'

files = os.listdir(path)

for file in files:
    print(file)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

shared_path = "/content/drive/Shareddrives"

print(os.path.exists(shared_path))